#### TODO

##### For the data, loop through all existent T-cell channels and segment them separately

In [1]:
from pathlib import Path
import numpy as np
import time
import shutil

import napari
from magicgui import magicgui
from magicgui.widgets import PushButton
from qtpy.QtWidgets import QPlainTextEdit, QWidget, QVBoxLayout, QApplication

from aicspylibczi import CziFile

from skimage import data, segmentation, feature, future
from skimage.measure import label
from skimage.segmentation import watershed, relabel_sequential

from sklearn.ensemble import RandomForestClassifier
from scipy.ndimage import binary_fill_holes, find_objects

from behav3d.utils.preprocessing import open_mask, dilate_mask
from behav3d.utils.segmentation import segment_size_filter, get_border_segments, remove_boundary_segments, calculate_edt, segment_2d_filter
from behav3d.utils.fileio import save_as_zarr, load_zarr, load_image, append_to_zarr

import multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from tqdm import tqdm

import joblib
from functools import partial

import zarr
import dask.array as da
import gc

## TODO create a function for BEHAV3D notebook


sigma_max = 16
features_func = partial(
        feature.multiscale_basic_features,
        intensity=True,
        edges=True,
        texture=True,
        # sigma_min=sigma_min,
        sigma_max=sigma_max,
        channel_axis=0,
    )

def postprocess_mask(mask, fill_holes=True, opening_nr_pixels=1):
    if fill_holes:  
        filled_mask = np.zeros_like(mask)
        for i in range(mask.shape[0]):
            filled_mask[i] = binary_fill_holes(mask[i])
        mask = filled_mask
    if opening_nr_pixels>0 and opening_nr_pixels is not None:
        mask = open_mask(mask)
    return(mask)


def refine_segment(args):
    """Refine a single segment by reapplying EDT-based splitting."""
    # start_time = time.time()
    # label_id, segment_mask, full_edt, edt_threshold, segment_size_min = args
    local_mask, local_edt, edt_thr_refined, segment_size_min, minc = args
    # if np.sum(segment_mask) == 0:
    #     return np.zeros_like(segment_mask)
   
    local_seeds = label(local_edt >= edt_thr_refined)
    # print("###", label_id, "refine_segment time elapsed: ", time.time() - start_time)
    if np.max(local_seeds) < 2:
        return (local_mask.astype(np.int32), tuple(minc))
    
    new_seg = watershed(-local_edt, markers=local_seeds, mask=local_mask)
    new_seg = segment_size_filter(new_seg, size_min=segment_size_min)
    new_seg = watershed(-local_edt, markers=new_seg, mask=local_mask)
    new_seg, _, _ = relabel_sequential(new_seg)
    return (new_seg, tuple(minc))

def segment_mask(
    mask, 
    segment_size_min=30, 
    use_dims=3, 
    n_workers=1
    ):
    offset = 1
    start_time = time.time()
    # Step 1: Initial segmentation
    edt = calculate_edt(mask, use_dims=use_dims)
    segments = label(mask)
    segments = segment_size_filter(segments, size_min=segment_size_min)
    segments = watershed(-edt, markers = segments, mask=mask)
    segments, _, _ = relabel_sequential(segments, offset)
    # Filter out 2D segments
    segments = segment_2d_filter(segments)
    return(segments)

# def segment_mask(
#     mask, 
#     edt_thr=1.5, 
#     edt_thr_refined=[2, 2.5, 3], 
#     segment_size_min=15, 
#     use_dims=3, 
#     n_workers=1
#     ):
#     offset = 1
#     start_time = time.time()
#     # Step 1: Initial segmentation
#     edt = calculate_edt(mask, use_dims=use_dims)
#     seeds = label(edt >= edt_thr)
#     segments = watershed(-edt, markers = seeds, mask=mask)
#     seeds2 = label(mask * (segments==0))
#     seeds2[seeds2!=0] += seeds.max()
#     # Relabel last segments to keep unique labels
#     segments[segments==0]=seeds2[segments==0]
#     segments = segment_size_filter(segments, size_min=segment_size_min)
#     segments = watershed(-edt, markers = segments, mask=mask)
#     # segments, _, _ = relabel_sequential(segments, offset)
    
#     # If edt_thr_refined ois not list, turn into list
#     if not isinstance(edt_thr_refined, list):
#         if edt_thr_refined is not None:
#             edt_thr_refined = [edt_thr_refined]
        
#     if edt_thr_refined is not None:
#         for thr in edt_thr_refined:
#             # Step 2: Prepare for per-label refinement
#             start_time = time.time()
#             unique_labels = np.unique(segments)
#             unique_labels = unique_labels[unique_labels != 0]  # Exclude background
#             args_list = []
    
#             slices = find_objects(segments)
#             for i, slc in enumerate(slices):
#                 label_id = i + 1  # Because label 0 is background and ignored

#                 if slc is None:
#                     continue  # This label is not present

#                 local_mask = segments[slc] == label_id
#                 local_edt = edt[slc]
#                 minc = [s.start for s in slc]

#                 args_list.append((local_mask, local_edt, thr, segment_size_min, minc))

#             if n_workers is None:
#                 n_workers = multiprocessing.cpu_count()

#             results = []
#             with ThreadPoolExecutor(max_workers=n_workers) as executor:
#                 results += list(executor.map(refine_segment, args_list))
           
#             current_label = 0
#             segments = np.zeros_like(mask, dtype=np.uint16)
#             for result in results:
#                 local_segment, minc = result
#                 z0, y0, x0 = minc
#                 z1, y1, x1 = z0 + local_segment.shape[0], y0 + local_segment.shape[1], x0 + local_segment.shape[2]

#                 # Assign to global array with label offset
#                 nonzero_mask = local_segment > 0
#                 local_segment[nonzero_mask] += current_label
#                 segments[z0:z1, y0:y1, x0:x1][nonzero_mask] = local_segment[nonzero_mask]

#                 current_label = segments.max() + 1  # Update for next segment
#     else:
#         segments, _, _ = relabel_sequential(segments, offset)
        
#     # FIlter out 2D segments
#     segments = segment_2d_filter(segments)
#     return(segments)

def segment_tcells(
    # mask_organoid,
    # mask_tcell,
    args,
    # tcell_edt_threshold_refined=[2, 2.5, 3],
    # tcell_edt_threshold_refined=None,
    tcell_segment_size_min=30,
    # organoid_edt_threshold=12,
    ):
   
    tcell_pixelclass, tcell_edt_threshold =  args
    
    tcell_channels = np.unique(tcell_pixelclass[tcell_pixelclass > 0])
    tcell_segments = []
    for tcell_ch in tcell_channels:
        tcell_mask = tcell_pixelclass == tcell_ch
        tcell_mask = postprocess_mask(tcell_mask, opening_nr_pixels=0)
        tcell_mask = segment_mask(
            mask=tcell_mask,
            segment_size_min=tcell_segment_size_min,
            # edt_thr=tcell_edt_threshold,
            # edt_thr_refined=tcell_edt_threshold_refined,
            use_dims=3
        )
        tcell_segments.append(tcell_mask)    
    
    combined_segments = np.zeros_like(tcell_segments[0], dtype=np.int32)
    
    first_label = 1
    for ch, tcell_seg in enumerate(tcell_segments):
        combined_segments[tcell_seg > 0] = tcell_seg[tcell_seg > 0 ] + first_label
        first_label = combined_segments.max() + 1
    # tcell_segments = np.stack(tcell_segments, axis=0)
    return(combined_segments)

def _apply_classifier(args):
    clf, path, outpath, idx = args
    features = np.asarray(load_image(path, mode="r")[idx])
    prediction = zarr.open(outpath, mode="r+")
    prediction[idx] = future.predict_segmenter(features, clf)

def apply_classifier(classifier, features_outpath, pred_labels_outpath, n_workers=4):
    shape = load_image(features_outpath).shape
    pred_labels = da.zeros(shape[:-1], chunks=(1,) + shape[1:-1], dtype='int16')
    save_as_zarr(pred_labels, pred_labels_outpath)
    args_list = [(classifier, str(features_outpath), str(pred_labels_outpath), idx) for idx in range(pred_labels.shape[0])]
    test=[]
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        test+=list(tqdm(executor.map(_apply_classifier, args_list), total=len(args_list)))
    
    pixel_class = load_image(pred_labels_outpath)
    pixel_class[pixel_class==1] = 0
    # pixel_class[pixel_class>0] -= 1
    pixel_class = pixel_class.compute()
    return pixel_class
        
def train_pixel_classifier(
    image_paths,
    output_dir,
    examples_per_sample = 3,
    n_workers=None
    ):

    
    if n_workers is None:
        n_workers = multiprocessing.cpu_count()
        
    pixel_class_outdir = Path(output_dir)
    pixel_class_outdir.mkdir(exist_ok=True, parents=True)
    
    features_outpath = Path(pixel_class_outdir, 'PixelClassifier_Features.zarr')
    image_outpath = Path(pixel_class_outdir, 'PixelClassifier_Images.zarr')
 
    if not features_outpath.exists() or not image_outpath.exists():
        if image_outpath.exists():
            shutil.rmtree(image_outpath)
        if features_outpath.exists():
            shutil.rmtree(features_outpath)
        
        all_images = []
        all_features = []
        for raw_image_path in image_paths:
            raw_image_path = Path(raw_image_path)
            sample_name = raw_image_path.stem
            print(f"Calculating features for: {sample_name}")
            raw_image_zarr =  Path(output_dir, "images", sample_name, f"{sample_name}.zarr")
            
            if not raw_image_zarr.exists():
                print(f"- Converting raw image to .zarr for memory efficiency...")
                images = load_image(raw_image_path)
                chunksize = (1,) + images.shape[1:]
                save_as_zarr(
                    img=images, 
                    path=raw_image_zarr, 
                    chunks=chunksize
                    )
                        
            images = load_image(raw_image_zarr)
            max_t = images.shape[0]-1
            print(images.shape)
            idc = np.linspace(0, max_t, examples_per_sample, dtype=int)
            print(f"Taking timepoints: {idc}")
            
            sample_images = [images[t] for t in idc]  
            
            all_images+=sample_images
            
            for img in tqdm(sample_images):
                append_to_zarr(np.expand_dims(features_func(img), axis=0), features_outpath)
        
        all_images = da.stack(all_images)
        all_images = all_images.transpose(1, 0, 2, 3, 4)
        save_as_zarr(all_images, image_outpath)
        del all_images
        gc.collect()
        
        all_images = load_zarr(image_outpath)
        all_features = load_zarr(features_outpath)
    else:
        all_images = load_image(image_outpath)
        all_features = load_image(features_outpath) 
            
    all_images = np.asarray(all_images)
    
    def segment_and_update(
        pixel_class_outdir,
        only_segment=False,
        tcell_edt_threshold=2.5,
        n_workers: int = 16,
        log=print
        ):
        start_time = time.time()
        log("###### Running Segmentation\n")
        QApplication.processEvents()
        # Access the label layer and feature image
        
        image_layer = viewer.layers['Image']
        image_data = image_layer.data
        
        tcell_label_layer = viewer.layers['User Provided Labels (Tcell)']
        tcell_label_data = tcell_label_layer.data
        
        tcell_labels_outpath = Path(pixel_class_outdir, 'PixelClassifier_UserTcellLabels.zarr')
        save_as_zarr(tcell_label_data, tcell_labels_outpath)
        
        def train_classifier(user_labels, features):
            flat_label_data = user_labels.ravel()
            flat_features = features.reshape(-1, features.shape[-1])  # shape: (N_total, 90)

            # Get 1D indices where labels exist
            label_indices = np.flatnonzero(flat_label_data > 0)

            selected_features = flat_features[label_indices].compute()  # (N_selected, 90)
            selected_labels = flat_label_data[label_indices]   
            
            nr_bg_pix = int(np.sum(selected_labels==1))
            nr_fg_pix = int(np.sum(selected_labels>1))
            total_pix = nr_bg_pix + nr_fg_pix
            
            log(f"Found {nr_bg_pix} background pixels")
            log(f"Found {nr_fg_pix} foreground pixels")

            unique_labels, counts = np.unique(selected_labels, return_counts=True)

            # Print each unique label and its count
            for label, count in zip(unique_labels, counts):
                log(f"Label {label}: {count} pixels")
            
            class_weights = {
                1: nr_bg_pix / total_pix,
                2: nr_fg_pix / total_pix,
            }
            clf = RandomForestClassifier(
                n_estimators=50,
                n_jobs=-1, 
                max_depth=20, 
                class_weight=class_weights
                )
            
            clf = future.fit_segmenter(selected_labels, selected_features, clf)
            return clf
        
        if not only_segment:           
            log("\n### Training Random Forest Classifier (T-cells)")
            clf_tcells = train_classifier(tcell_label_data, all_features)
            tcell_random_forest_outpath = Path(pixel_class_outdir, 'PixelClassifier_Tcell.joblib')
            log("Saving RandomForest, Sparse labels and input images to {tcell_random_forest_outpath}")
            joblib.dump(clf_tcells, tcell_random_forest_outpath)
            QApplication.processEvents()
        
        pred_tcell_labels_outpath = Path(pixel_class_outdir, 'PixelClassifier_Tcell_PredictedLabels.zarr')

        if not only_segment:
            if pred_tcell_labels_outpath.exists():
                shutil.rmtree(pred_tcell_labels_outpath)
                
            log("\n### Predicting T-cell Pixels")
            QApplication.processEvents()
            pred_tcell_mask = apply_classifier(clf_tcells, features_outpath, pred_tcell_labels_outpath)
            
            viewer.layers["Pixel Classification (Tcell)"].data = pred_tcell_mask

        else:      
            log("\n### Loading T-cell Prediction Mask")
            QApplication.processEvents()
            pred_tcell_mask = viewer.layers["Pixel Classification (Tcell)"].data
            pred_tcell_mask[pred_tcell_mask==1] = 0
            viewer.layers["Pixel Classification (Tcell)"].data = pred_tcell_mask
            
        log("\n### Segment T-Cell instances")
        QApplication.processEvents()
        args_list = [(pred_tcell_mask[idx], tcell_edt_threshold) for idx in range(tcell_label_data.shape[0])]
        results=[]

        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            results+=list(tqdm(executor.map(segment_tcells, args_list), total=len(args_list)))
        full_seg_tcell = np.stack(results, axis=0)
        
        log("\n### Updating Napari Data")
        
        # tcell_channels = np.unique(pred_tcell_mask[pred_tcell_mask > 0])
        # for idx, tcell_ch in enumerate(tcell_channels):
        #     layer_name = f"Tcell Segments (Channel {tcell_ch})"
        if "Tcell Segments" not in viewer.layers:
            viewer.add_labels(
                full_seg_tcell, 
                name="Tcell Segments", 
                opacity=0.7, 
                visible=False
            )
        else:
            viewer.layers["Tcell Segments"].data = full_seg_tcell
        
        image_outpath = Path(pixel_class_outdir, 'PixelClassifier_Images.zarr')
        save_as_zarr(image_data, image_outpath)
        
        log(f"\n###### DONE time elapsed: {time.time() - start_time:.2f} s")
    
    def save_pixel_classification(log=print):
        label_layer = viewer.layers['User Provided Labels']
        label_data = label_layer.data

        labels_outpath = Path(pixel_class_outdir, 'PixelClassifier_UserLabels.zarr')
        save_as_zarr(label_data, labels_outpath)
        log(f"Saved Pixel Classification to: {labels_outpath}")
    
    print(all_images.shape)
    # Create Napari viewer
    viewer = napari.Viewer()
    img_layer = viewer.add_image(
        all_images, 
        name="Image", 
        contrast_limits=(0,float(np.percentile(all_images[1,-1].reshape(-1), 99.99)))
        )
    img_layer.contrast_limits_range = (0, all_images.max())

    for ch, ch_img in enumerate(all_images):
        viewer.add_image(
            ch_img, 
            name=f"Image Channel {ch+1}", 
            visible=False,
            )
        
    img_layer = viewer.add_image(
        np.transpose(all_images[[1,2,3]], (1, 2, 3, 4, 0)), 
        name="Image RGB", 
        rgb=True
        )
    img_layer.contrast_limits_range = (0, all_images.max())
    
    tcell_labels_outpath = Path(pixel_class_outdir, 'PixelClassifier_UserTcellLabels.zarr')
        
    if tcell_labels_outpath.exists():
        print("Loading existing user labelled Tcell data")
        tcell_user_labels = np.asarray(load_zarr(tcell_labels_outpath))
    else:
        tcell_user_labels = np.zeros(all_images.shape[1:]).astype(np.int16)
  
    user_layers = {
        "User Provided Labels (Tcell)": tcell_user_labels,
    }

    for name, data in user_layers.items():
        layer = viewer.add_labels(data, name=name, opacity=0.3)
    
    pred_tcell_labels_outpath = Path(pixel_class_outdir, 'PixelClassifier_Tcell_PredictedLabels.zarr')
    if not pred_tcell_labels_outpath.exists():
        pixelclass_layers = {
            "Pixel Classification (Tcell)": np.zeros(all_images.shape[1:]).astype(np.int16),
        }
    else:
        loaded_data = load_zarr(pred_tcell_labels_outpath)
        loaded_data[loaded_data==1] = 0  # Set background to 0
        pixelclass_layers = {
            "Pixel Classification (Tcell)": np.asarray(loaded_data),
        }
    
    for name, data in pixelclass_layers.items():
        layer = viewer.add_labels(data, name=name, opacity=0.3, visible=False)
   
    # viewer.add_labels(np.zeros(all_images.shape[1:]).astype(np.int16), name="Tcell Segments", opacity=0.7, visible=False)

    log_output = QPlainTextEdit()
    log_output.setReadOnly(True)
    log_widget = QWidget()
    layout = QVBoxLayout()
    layout.addWidget(log_output)
    log_widget.setLayout(layout)
    viewer.window.add_dock_widget(log_widget, area="right", name="Log Output")
    
    update_function = partial(
        segment_and_update, 
        pixel_class_outdir=pixel_class_outdir,
        n_workers=n_workers,
        log=log_output.appendPlainText
        )
    gui = magicgui(update_function, 
                tcell_edt_threshold={"widget_type": "FloatSlider", "min": 1.0, "max": 15.0, "step": 0.5},
                only_segment={"widget_type": "Checkbox", "text": "Only Segment"}
                )
    viewer.window.add_dock_widget(gui)
        
    save_button = PushButton(label="Save User Labels")
    save_function = partial(
        save_pixel_classification,
        log =log_output.appendPlainText
    )
    save_button.clicked.connect(save_function)
    gui.native.layout().addWidget(save_button.native)

    napari.run()


def _run_single_timepoint_segmentation(
    t_img,
    clf_tcell,
    tcell_edt_threshold=2,
    ):
    features = features_func(t_img)

    pred_tcell_mask = future.predict_segmenter(features, clf_tcell)
    
    pred_tcell_mask[pred_tcell_mask>0] -= 1
    
    seg_tcell = segment_tcells(
        args = (pred_tcell_mask, tcell_edt_threshold),  
    )
    return(seg_tcell)
    
def run_pixel_classifier_segmentation(
    image_paths,
    output_dir,
    tcell_edt_threshold=2
    ):
    clf_tcell_path = Path(output_dir, 'PixelClassifier_Tcell.joblib')
    clf_tcell = joblib.load(clf_tcell_path)
    
    for raw_image_path in image_paths:
        raw_image_path = Path(raw_image_path)
        sample_name = raw_image_path.stem
        raw_image_zarr =  Path(output_dir, "images", sample_name, f"{sample_name}.zarr")
        img_outdir = Path(output_dir, "images", sample_name)
        if not img_outdir.exists():
            img_outdir.mkdir(parents=True)
            
        tcell_segments_outpath = Path(img_outdir, f"{sample_name}_tcell_segments.zarr")
        
        if not raw_image_zarr.exists():
            img = load_image(raw_image_path)
            save_as_zarr(img, raw_image_zarr)
        img = load_image(raw_image_zarr)
        print(img.shape)
        
        if tcell_segments_outpath.exists():
            print("Already segmented, skipping")
        else:   
            if tcell_segments_outpath.exists():
                shutil.rmtree(tcell_segments_outpath)
            for t, t_img in tqdm(enumerate(img), total=img.shape[0]):
                seg_tcell = _run_single_timepoint_segmentation(
                    t_img=t_img,
                    clf_tcell=clf_tcell,
                    tcell_edt_threshold=tcell_edt_threshold
                )
                append_to_zarr(np.expand_dims(seg_tcell, axis=0), tcell_segments_outpath)
                

## Set paths

In [2]:
# output_dir = "/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/segmentation"
# image_paths = [
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img02_W1-1_30s.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img04_W1-2_30s.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img05_W1-3_30s.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img07_W1-9_2min.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img09_W1-6_2min.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img12_W1-7_2min.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img13_W1-5_2min.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img15_W1-8_2min.czi',
#         '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/split_colors/BHVD_JM1_Exp001_Img17_W1-4_30s.czi'
# ]

output_dir = "/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/segmentation"
image_paths = [
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img01_W2-1_30s.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img03_W2-2_30s.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img06_W2-3_30s.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img08_W2-7_2min.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img10_W2-6_2min.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img11_W2-9_2min.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img14_W2-8_2min.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img16_W2-5_30s.czi',
        '/Volumes/T7_Sam/BHVD_BEHAV3D/BEHAV3D_python/data/multicolor_tcell_GT/LSM980/combined_colors/BHVD_JM1_Exp001_Img18_W2-4_30s.czi'
        ]


## Train Classifier

In [ ]:
examples_per_sample = 3

train_pixel_classifier(
    image_paths=image_paths,
    output_dir=output_dir,
    examples_per_sample=examples_per_sample,
    n_workers=8
)

(4, 27, 23, 400, 400)
Loading existing user labelled Tcell data


100%|██████████| 27/27 [00:10<00:00,  2.57it/s]
Traceback (most recent call last):
  File "/Users/s.deblank-3/miniforge3/envs/behav3d/lib/python3.12/site-packages/napari/_qt/layer_controls/qt_layer_controls_base.py", line 177, in changeOpacity
    self.layer.opacity = value
    ^^^^^^^^^^^^^^^^^^
  File "/Users/s.deblank-3/miniforge3/envs/behav3d/lib/python3.12/site-packages/napari/layers/base/base.py", line 748, in opacity
    self._update_thumbnail()
  File "/Users/s.deblank-3/miniforge3/envs/behav3d/lib/python3.12/site-packages/napari/layers/labels/labels.py", line 971, in _update_thumbnail
    downsampled = ndi.zoom(image, zoom_factor, prefilter=False, order=0)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/s.deblank-3/miniforge3/envs/behav3d/lib/python3.12/site-packages/scipy/ndimage/_interpolation.py", line 813, in zoom
    zoom = _ni_support._normalize_sequence(zoom, input.ndim)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [4]:
run_pixel_classifier_segmentation(
    image_paths=image_paths,
    output_dir=output_dir,
    )



(720, 4, 23, 400, 400)
Already segmented, skipping
(720, 4, 23, 400, 400)


100%|██████████| 720/720 [3:20:54<00:00, 16.74s/it]  


(720, 4, 23, 400, 400)


100%|██████████| 720/720 [3:20:00<00:00, 16.67s/it]  


(180, 4, 23, 400, 400)


...

100%|██████████| 180/180 [49:58<00:00, 16.66s/it]


(218, 4, 23, 400, 400)


100%|██████████| 218/218 [1:00:29<00:00, 16.65s/it]


(218, 4, 23, 400, 400)


100%|██████████| 218/218 [1:00:37<00:00, 16.69s/it]
